In [ ]:
import os
import torch
import telebot
from transformers import AutoModelForCausalLM, AutoTokenizer

TELEGRAM_TOKEN = "8705798467:AAFJeacrrH3R3-UlT4M3ZOF_wEt9Bebg7Kk"
LLM_PATH = "FP-KCV/jawani-sealion-gatra-2-9b"

tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)
model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

bot = telebot.TeleBot(TELEGRAM_TOKEN)

@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    bot.reply_to(message, "Woi! Ana sing bisa dibantu ra dina iki?")

@bot.message_handler(func=lambda message: True)
def handle_message(message):
    user_query = message.text
    chat_id = message.chat.id
    
    bot.send_chat_action(chat_id, 'typing')

    system_instruction = "Kowe dadi kanca cerak sing asik. Gunakake basa Ngoko Lugu sing santai banget. Aja kaku-kaku."
    prompt = f"Instruksi: {system_instruction}\nKanca: {user_query}\nAI:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    try:
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs, 
                max_new_tokens=256,
                temperature=0.8,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        full_response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        if "AI:" in full_response:
            clean_response = full_response.split("AI:")[-1].strip()
        else:
            clean_response = full_response.replace(prompt, "").strip()
            
        bot.reply_to(message, clean_response)
        
    except Exception as e:
        print(f"Error: {e}")
        bot.reply_to(message, "Waduh, lagi error ki servere.")

bot.infinity_polling()